# Lab 9: Πολυπρακτορικά Συστήματα με CrewAI

## Σκοπός

Σε αυτό το εργαστήριο θα **κατασκευάσουμε ένα πολυπρακτορικό σύστημα** που προσομοιώνει μια μικρή αναλυτική ομάδα επενδύσεων. Θα δούμε πρακτικά:

- Πώς ορίζονται **AI Agents** με ρόλους, στόχους και ιστορικό (backstory)
- Πώς αναθέτουμε **Tasks** σε συγκεκριμένους πράκτορες
- Πώς μια **Crew** συντονίζει τους πράκτορες και παράγει ένα τελικό αποτέλεσμα
- Πώς η **αλυσίδα επικοινωνίας** μεταξύ πρακτόρων παράγει έξοδο που κανένας μεμονωμένος agent δεν θα παρήγαγε μόνος του

## Σενάριο

Φανταστείτε ότι εργάζεστε σε ένα μικρό venture capital fund. Ο manager σας θέλει μια **γρήγορη αναφορά** για μια εταιρεία πριν από μια επενδυτική συνάντηση. Δεν υπάρχει χρόνος να γίνει χειροκίνητη έρευνα — αναθέτουμε τη δουλειά σε μια **AI Crew**:

| Agent | Ρόλος | Αρμοδιότητα |
|---|---|---|
| **Researcher** | Senior Market Analyst | Συλλέγει πληροφορίες για την εταιρεία |
| **Analyst** | Investment Advisor | Γράφει την τελική αναφορά υπέρ/κατά |

---

## 🔑 Προαπαιτούμενο: Δωρεάν API Key από το Groq

Χρησιμοποιούμε το **Groq** ως LLM provider — είναι **δωρεάν** και δεν χρειάζεται πιστωτική κάρτα.

1. Πηγαίνετε στο [console.groq.com](https://console.groq.com)
2. Δημιουργήστε λογαριασμό (με Google ή email)
3. Πηγαίνετε στο **API Keys** → **Create API Key**
4. Αντιγράψτε το key (ξεκινάει με `gsk_...`)
5. Επικολλήστε το στο κελί παρακάτω


In [ ]:
# Εγκατάσταση βιβλιοθηκών (εκτελέστε μία φορά)
# Αν εκτελείτε σε Google Colab ή τοπικό περιβάλλον:
%pip install -q crewai crewai-tools

In [ ]:
import os

# ✏️ ΒΑΛΤΕ ΕΔΩ ΤΟ GROQ API KEY ΣΑΣ
os.environ["GROQ_API_KEY"] = "gsk_ΑΝΤΙΚΑΤΑΣΤΗΣΤΕ_ΜΕ_ΤΟ_ΔΙΚΟ_ΣΑΣ_KEY"

# Επαλήθευση ότι το key έχει οριστεί
key = os.environ.get("GROQ_API_KEY", "")
if key.startswith("gsk_") and len(key) > 20:
    print("✅ Groq API Key ορίστηκε επιτυχώς.")
else:
    print("❌ Το API Key δεν φαίνεται σωστό. Ελέγξτε ότι ξεκινά με gsk_")

## Βήμα 1: Ορισμός του LLM

Το CrewAI χρειάζεται έναν **"εγκέφαλο"** για κάθε agent — εδώ χρησιμοποιούμε το **Llama 3.3 70B** μέσω Groq.

Η κλάση `LLM` του CrewAI επικοινωνεί με τον provider (Groq) χρησιμοποιώντας το πρωτόκολλο LiteLLM. Το format του model name είναι: `"provider/model-name"`.


In [ ]:
from crewai import LLM

# Ορίζουμε το LLM που θα χρησιμοποιηθεί από όλους τους agents
# Διαθέσιμα δωρεάν μοντέλα Groq: llama-3.3-70b-versatile, llama-3.1-8b-instant, gemma2-9b-it
llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0.7  # Λίγη δημιουργικότητα, αλλά όχι υπερβολική
)

print("✅ LLM ορίστηκε:", llm.model)

## Βήμα 2: Ορισμός των Πρακτόρων (Agents)

Κάθε `Agent` ορίζεται από τρία βασικά στοιχεία:

| Παράμετρος | Τι κάνει |
|---|---|
| `role` | Ο τίτλος/ρόλος του agent — επηρεάζει τον τρόπο που «σκέφτεται» |
| `goal` | Ο συγκεκριμένος στόχος που πρέπει να επιτύχει |
| `backstory` | Το «ιστορικό» του — δίνει πλαίσιο και προσωπικότητα |

> 💡 **Γιατί το backstory;** Τα LLMs αποδίδουν καλύτερα όταν έχουν πλαίσιο. Το backstory είναι essentially **prompt engineering** — πείθουμε το μοντέλο να «παίξει» έναν ρόλο.


In [ ]:
from crewai import Agent

# ── Agent 1: Ερευνητής ──────────────────────────────────────────────────────
researcher = Agent(
    role="Senior Market Research Analyst",
    goal=(
        "Συλλέξε ολοκληρωμένες πληροφορίες για την εταιρεία {company}: "
        "κύρια προϊόντα/υπηρεσίες, θέση στην αγορά, πρόσφατες εξελίξεις, "
        "ανταγωνισμός και βασικά οικονομικά στοιχεία."
    ),
    backstory=(
        "Είσαι έμπειρος αναλυτής με 15 χρόνια στον κλάδο της τεχνολογίας. "
        "Ειδικεύεσαι στην ανάλυση εταιρειών υψηλής τεχνολογίας και έχεις εργαστεί "
        "σε κορυφαία investment banks. Η ανάλυσή σου είναι πάντα δομημένη, "
        "αντικειμενική και βασισμένη σε στοιχεία."
    ),
    llm=llm,
    verbose=True  # Εμφάνιση της σκέψης του agent σε πραγματικό χρόνο
)

# ── Agent 2: Αναλυτής/Σύμβουλος ─────────────────────────────────────────────
advisor = Agent(
    role="Investment Advisor",
    goal=(
        "Με βάση την έρευνα που έχει γίνει, γράψε μια επαγγελματική αναφορά "
        "επενδυτικής αξιολόγησης για την εταιρεία {company}. "
        "Η αναφορά πρέπει να περιλαμβάνει σαφή θέση (Αγορά / Αναμονή / Πώληση)."
    ),
    backstory=(
        "Είσαι senior investment advisor με εξειδίκευση στον τεχνολογικό τομέα. "
        "Γράφεις αναφορές για θεσμικούς επενδυτές και portfolio managers. "
        "Η γλώσσα σου είναι επαγγελματική αλλά κατανοητή. "
        "Πάντα παρουσιάζεις τα υπέρ και τα κατά πριν δώσεις σύσταση."
    ),
    llm=llm,
    verbose=True
)

print("✅ Δύο agents ορίστηκαν:")
print(f"  1. {researcher.role}")
print(f"  2. {advisor.role}")

## Βήμα 3: Ορισμός Εργασιών (Tasks)

Κάθε `Task` αντιστοιχεί σε μια συγκεκριμένη εργασία που αναθέτουμε σε έναν agent.

Σημαντική διαφορά από ένα απλό prompt:
- Το `description` περιγράφει **τι πρέπει να γίνει**
- Το `expected_output` ορίζει **ακριβώς τι αναμένουμε** — λειτουργεί σαν acceptance criteria
- Το `context` μεταφέρει **το output της προηγούμενης task** στον επόμενο agent


In [ ]:
from crewai import Task

# Η εταιρεία που θα αναλυθεί — αλλάξτε την για να δοκιμάσετε διαφορετικά!
COMPANY = "NVIDIA"

# ── Task 1: Έρευνα ───────────────────────────────────────────────────────────
research_task = Task(
    description=(
        f"Κάνε εκτενή ανάλυση της εταιρείας {COMPANY}. "
        "Κάλυψε τους παρακάτω τομείς:\n"
        "1. Επισκόπηση εταιρείας (ίδρυση, μέγεθος, τομέας)\n"
        "2. Κύρια προϊόντα και υπηρεσίες\n"
        "3. Ανταγωνιστικό πλεονέκτημα (moat)\n"
        "4. Κύριοι ανταγωνιστές\n"
        "5. Πρόσφατες εξελίξεις και τάσεις στον κλάδο"
    ),
    expected_output=(
        "Δομημένη αναφορά έρευνας σε 5 ενότητες με bullet points. "
        "Μέγιστο 400 λέξεις. Χρησιμοποίησε αγγλικούς όρους για τα τεχνικά/χρηματοοικονομικά."
    ),
    agent=researcher  # Αυτός ο agent είναι υπεύθυνος
)

# ── Task 2: Επενδυτική Αναφορά ───────────────────────────────────────────────
analysis_task = Task(
    description=(
        f"Με βάση την έρευνα για την {COMPANY}, "
        "ετοίμασε μια επενδυτική αναφορά για έναν portfolio manager. "
        "Δομή αναφοράς:\n"
        "- Executive Summary (2-3 προτάσεις)\n"
        "- Ισχυρά σημεία (Υπέρ επένδυσης)\n"
        "- Κίνδυνοι (Κατά επένδυσης)\n"
        "- Τελική Σύσταση: ΑΓΟΡΑ / ΑΝΑΜΟΝΗ / ΠΩΛΗΣΗ με αιτιολόγηση"
    ),
    expected_output=(
        "Επαγγελματική επενδυτική αναφορά σε 4 ενότητες. "
        "Η τελική σύσταση να είναι ξεκάθαρη και αιτιολογημένη."
    ),
    agent=advisor,
    context=[research_task]  # Ο advisor βλέπει το output του researcher
)

print(f"✅ Δύο tasks ορίστηκαν για: {COMPANY}")
print(f"  Task 1 → {researcher.role}")
print(f"  Task 2 → {advisor.role} (με context από Task 1)")

## Βήμα 4: Δημιουργία της Crew και Εκτέλεση

Η `Crew` είναι ο **ορχηστράτορας** — συντονίζει τους agents και τα tasks.

Υπάρχουν δύο τρόποι εκτέλεσης (`Process`):
- **`sequential`**: Tasks εκτελούνται σειριακά (ένα μετά το άλλο) — απλό και προβλέψιμο
- **`hierarchical`**: Ένας manager agent κατανέμει εργασίες δυναμικά — πιο ευέλικτο

Εδώ χρησιμοποιούμε `sequential` για να δούμε ξεκάθαρα τη ροή: **Researcher → Advisor**.


In [ ]:
from crewai import Crew, Process

# Δημιουργία της Crew
investment_crew = Crew(
    agents=[researcher, advisor],
    tasks=[research_task, analysis_task],
    process=Process.sequential,  # Researcher πρώτα, μετά Advisor
    verbose=True
)

print("✅ Crew δημιουργήθηκε.")
print(f"   Agents: {len(investment_crew.agents)}")
print(f"   Tasks:  {len(investment_crew.tasks)}")
print(f"   Process: Sequential")
print("\n" + "="*60)
print("🚀 Εκκίνηση... (αναμένετε 30-60 δευτερόλεπτα)")
print("="*60 + "\n")

# Εκτέλεση — kickoff() ξεκινά τη ροή
result = investment_crew.kickoff(inputs={"company": COMPANY})

In [ ]:
# Εμφάνιση τελικού αποτελέσματος
print("\n" + "="*60)
print(f"📊 ΤΕΛΙΚΗ ΑΝΑΦΟΡΑ: {COMPANY}")
print("="*60)
print(result.raw)

## Παρατηρήσεις & Ερωτήσεις για Συζήτηση

Αφού εκτελέσετε το notebook, σκεφτείτε τα παρακάτω:

**1. Παρατήρηση στο verbose output:**
Τι είδατε στην έξοδο ενώ έτρεχαν οι agents; Πώς «σκεφτόταν» ο καθένας;

**2. Σύνδεση με τη θεωρία:**
- Ποια είναι τα **Beliefs** του Researcher; (= η γνώση του LLM για την εταιρεία)
- Ποιο είναι το **Goal/Intention** του Advisor;
- Ποιο είναι το «εργαλείο» (Tool) που χρησιμοποιεί ο Advisor; (= το context από τον Researcher)

**3. Αρχιτεκτονική:**
Αυτό είναι **Sequential** process. Σε ποια περίπτωση θα χρησιμοποιούσατε Hierarchical;

**4. Περιορισμοί:**
Τι αδυναμία βλέπετε στο σύστημα αυτό; (Hint: τα δεδομένα είναι πρόσφατα;)


## ✏️ Άσκηση: Δοκιμάστε Διαφορετικές Εταιρείες & Ρόλους

Τροποποιήστε τον παρακάτω κώδικα για να:
1. Αναλύσετε μια ελληνική εταιρεία (π.χ. OTE, Jumbo, ΔΕΗ)
2. Προσθέσετε έναν τρίτο agent: **Risk Officer** που αξιολογεί τους κινδύνους
3. Αλλάξετε τη `temperature` και παρατηρήστε αν η αναφορά αλλάζει


In [ ]:
# ✏️ ΑΣΚΗΣΗ: Αλλάξτε την εταιρεία ή προσθέστε τρίτο agent

# Βήμα 1: Αλλάξτε εταιρεία
MY_COMPANY = "Apple"  # ← Αλλάξτε εδώ

# Βήμα 2 (Προαιρετικό): Προσθέστε Risk Officer
risk_officer = Agent(
    role="Chief Risk Officer",
    goal=f"Εντόπισε και αξιολόγησε τους κύριους κινδύνους επένδυσης στην {MY_COMPANY}.",
    backstory=(
        "Ειδικεύεσαι στην ανάλυση ρίσκου επενδύσεων. "
        "Εντοπίζεις κινδύνους που άλλοι παραβλέπουν: "
        "ρυθμιστικούς, τεχνολογικούς, ανταγωνιστικούς και μακροοικονομικούς."
    ),
    llm=llm,
    verbose=True
)

risk_task = Task(
    description=(
        f"Με βάση την ανάλυση για την {MY_COMPANY}, "
        "κατάρτισε έναν πίνακα κινδύνων (Risk Matrix) με τους top-5 κινδύνους. "
        "Για κάθε κίνδυνο: Πιθανότητα (Υψηλή/Μέτρια/Χαμηλή) και Επίπτωση (Υψηλή/Μέτρια/Χαμηλή)."
    ),
    expected_output="Πίνακας 5 κινδύνων με πιθανότητα και επίπτωση.",
    agent=risk_officer,
    context=[research_task]  # Χρησιμοποιεί την ίδια έρευνα
)

# Νέα Crew με τρεις agents
extended_crew = Crew(
    agents=[researcher, advisor, risk_officer],
    tasks=[research_task, analysis_task, risk_task],
    process=Process.sequential,
    verbose=True
)

print(f"🚀 Εκτέλεση με {len(extended_crew.agents)} agents για: {MY_COMPANY}")
# Αποσχολιάστε την επόμενη γραμμή για να τρέξετε:
# result2 = extended_crew.kickoff(inputs={"company": MY_COMPANY})
# print(result2.raw)

---

## Σύνοψη

| Έννοια | Υλοποίηση στο CrewAI |
|---|---|
| **Agent** = αυτόνομη οντότητα με στόχο | `Agent(role, goal, backstory, llm)` |
| **Task** = συγκεκριμένη ανάθεση | `Task(description, expected_output, agent)` |
| **Context** = επικοινωνία μεταξύ agents | `Task(context=[άλλο_task])` |
| **Crew** = ορχηστράτορας ομάδας | `Crew(agents, tasks, process)` |
| **Sequential Process** = αλυσίδα | Agent A → Agent B → Agent C |

### Περαιτέρω Εξερεύνηση

- **Groq models**: `llama-3.1-8b-instant` (πιο γρήγορο), `gemma2-9b-it` (Google)
- **Web search**: Προσθέστε `SerperDevTool` ή `DuckDuckGoSearchRun` για πραγματική αναζήτηση
- **Hierarchical process**: Ένας manager agent κατανέμει εργασίες αυτόματα
- **Memory**: Οι agents μπορούν να «θυμούνται» προηγούμενες συνεδρίες
- **Docs**: [docs.crewai.com](https://docs.crewai.com)
